In [4]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
np.random.seed(109204)
plt.rcParams.update({'axes.labelsize': 18, 'xtick.labelsize': 16, 'ytick.labelsize': 16, 'axes.titlesize': 18,})

def henon_map(x, y, a, b): # Function for Henon
    return 1 - a*x**2 + y, b*x

sigma_Q = 0.5  # Controls Variability of Q_k
def ukf_henon_estimation(a_true, a_guess, b_true, b_guess, n_steps=100, Q_error=1e-5, x0_true=0.1, x0_hat=0.1, y0_true = 0.1, y0_hat = 0.1, meas_noise=None, alpha=1e-3, beta=2, kappa=0):
    n = 4
    lam = alpha**2 * (n + kappa) - n # Lambda substitution
    c = n + lam # Denominator of w
    Wm = np.full(2*n + 1, 1/(2*c)) # For Mean
    Wc = np.full(2*n + 1, 1/(2*c)) # For Variance
    Wm[0] = lam/c
    Wc[0] = (-alpha**2 + beta) + lam/c
    
    generated = False
    Q0 = Q_error
    Q_seq = Q0 * np.exp(sigma_Q * np.random.randn(n_steps))
    if meas_noise is None:
        generated = True
        x_true = np.zeros(n_steps)
        y_true = np.zeros(n_steps)
        x_meas = np.zeros(n_steps)
        y_meas = np.zeros(n_steps)
        x_true[0] = x0_true
        y_true[0] = y0_true
        x_meas[0] = x_true[0] + np.random.normal(0, np.sqrt(Q_seq[0]))
        y_meas[0] = y_true[0] + np.random.normal(0, np.sqrt(Q_seq[0]))
        for k in range(1, n_steps):
            x_true[k], y_true[k] = henon_map(x_true[k-1], y_true[k-1], a_true, b_true)  
            x_meas[k] = x_true[k] + np.random.normal(0, np.sqrt(Q_seq[k]))
            y_meas[k] = y_true[k] + np.random.normal(0, np.sqrt(Q_seq[k]))
    else:
        x_true = None # If data does already exist, just use that
        y_true = None

    x_hat = np.zeros(n_steps)
    y_hat = np.zeros(n_steps)
    a_hat = np.zeros(n_steps)
    b_hat = np.zeros(n_steps)
    x_hat[0] = x0_hat
    y_hat[0] = y0_hat
    a_hat[0] = a_guess    
    b_hat[0] = b_guess
    P = np.eye(n)
    Q = Q_error * np.eye(2) # Make the error 2-dimensional

    def sigma_points(mu, P): # Sigma Point Generator
        U, s, _ = np.linalg.svd(c * P, full_matrices=False)
        s = np.maximum(s, 1e-12) # SVD
        A = U @ np.diag(np.sqrt(s))
        SP = np.zeros((2*n + 1, n))
        SP[0] = mu    
        
        for i in range(n):
            SP[i + 1] = mu + A[:, i]
            SP[n + i + 1] = mu - A[:, i]         
        return SP

    for k in range(1, n_steps): # UKF Loop
        mu = np.array([x_hat[k-1], y_hat[k-1], a_hat[k-1], b_hat[k-1]])

        X = sigma_points(mu, P) # Form Sigma Points
        
        X_pred = np.zeros_like(X) # Propagate SP through Phi
        for i in range(2 * n + 1):
            x_i, y_i, a_i, b_i = X[i]
            x_next, y_next = henon_map(x_i, y_i, a_i, b_i)
            a_next, b_next = a_i, b_i  # Assume constant a,b
            X_pred[i] = np.array([x_next, y_next, a_next, b_next])

        mu_pred = np.sum(Wm[:, None] * X_pred, axis=0) # Predict Mean
        
        P_pred = np.zeros((n, n)) # Predict Covariance
        for i in range(2 * n + 1):
            d = (X_pred[i] - mu_pred).reshape(-1, 1)
            P_pred += Wc[i] * (d @ d.T)

        m = 2
        Y_sigma = np.zeros((2*n + 1, m))
        for i in range(2*n + 1):
            Y_sigma[i] = X_pred[i, 0:2]

        y_pred = np.sum(Wm[:, None] * Y_sigma, axis=0) # Measurement for Mean

        S = np.zeros((m, m)) # Measurement for Variance Covariance
        for i in range(2*n + 1):
            dy = (Y_sigma[i] - y_pred).reshape(-1, 1)
            S += Wc[i] * (dy @ dy.T)
        S += Q_seq[k] * np.eye(2)

        P_xy = np.zeros((n, m)) # Cross-Covariance between State & Measurement
        for i in range(2 * n + 1):
            dx = (X_pred[i] - mu_pred).reshape(-1, 1)
            dy = (Y_sigma[i] - y_pred).reshape(1, -1)
            P_xy += Wc[i] * (dx @ dy)

        K = P_xy @ np.linalg.inv(S)
        z_k = np.array([x_meas[k], y_meas[k]])
        innovation = z_k - y_pred
        mu_upd = mu_pred + K @ innovation
        P_upd = P_pred - K @ S @ K.T
        x_hat[k], y_hat[k], a_hat[k], b_hat[k] = mu_upd[0], mu_upd[1], mu_upd[2], mu_upd[3]
        P = P_upd        
    return x_true, y_true, x_meas, y_meas, x_hat, y_hat, a_hat, b_hat

def interactive_ukf_henon(a_true=1.4, b_true=0.3, Q_error=1e-5, n_steps=100, alpha=1e-3, beta=2, kappa=0):
    plt.close('all')
    a_guess, b_guess = 1.0, 0.1
    n_runs = 100

    Q0 = Q_error
    Q_seq = Q0 * np.exp(sigma_Q * np.random.randn(n_steps)) 
    x_true_base = np.zeros(n_steps) # True Henon Map (Single Trajectory)
    y_true_base = np.zeros(n_steps)
    x_meas_base = np.zeros(n_steps)
    y_meas_base = np.zeros(n_steps)
    x_true_base[0] = 0.7
    y_true_base[0] = 0.7
    x_meas_base[0] = x_true_base[0] + np.random.normal(0, np.sqrt(Q_seq[0]))
    y_meas_base[0] = y_true_base[0] + np.random.normal(0, np.sqrt(Q_seq[0]))
    for k in range(1, n_steps):
        x_true_base[k], y_true_base[k] = henon_map(x_true_base[k-1], y_true_base[k-1], a_true, b_true)
        x_meas_base[k] = x_true_base[k] + np.random.normal(0, np.sqrt(Q_seq[k]))
        y_meas_base[k] = y_true_base[k] + np.random.normal(0, np.sqrt(Q_seq[k]))
    
    all_x_true = np.zeros((n_runs, n_steps))
    all_y_true = np.zeros((n_runs, n_steps))
    all_x_hat = np.zeros((n_runs, n_steps))
    all_y_hat = np.zeros((n_runs, n_steps))
    all_a_hat = np.zeros((n_runs, n_steps))
    all_b_hat = np.zeros((n_runs, n_steps))

    for i in range(n_runs):
        x_true, y_true, x_meas, y_meas, x_hat, y_hat, a_hat, b_hat = ukf_henon_estimation(
            a_true=a_true,
            b_true=b_true,
            a_guess=a_guess,
            b_guess=b_guess,
            n_steps=n_steps,
            Q_error=Q_error, 
            meas_noise = None)
        all_x_true[i, :] = x_true_base
        all_y_true[i, :] = y_true_base
        all_x_hat[i, :] = x_hat
        all_y_hat[i, :] = y_hat
        all_a_hat[i, :] = a_hat
        all_b_hat[i, :] = b_hat

    x_hat_mean = np.mean(all_x_hat, axis=0) # Statistics from MC
    y_hat_mean = np.mean(all_y_hat, axis=0)
    x_hat_p05 = np.percentile(all_x_hat, 5, axis=0)
    x_hat_p95 = np.percentile(all_x_hat, 95, axis=0)
    y_hat_p05 = np.percentile(all_y_hat, 5, axis=0)
    y_hat_p95 = np.percentile(all_y_hat, 95, axis=0)

    mse_x = np.mean((all_x_hat - all_x_true)**2, axis=0) # MSE of MC
    variance_x = np.var(all_x_hat, axis=0) # MC Variance (Filter Spread)
    mse_y = np.mean((all_y_hat - all_y_true)**2, axis=0)
    variance_y = np.var(all_y_hat, axis=0)

    a_hat_mean = np.mean(all_a_hat, axis=0)
    b_hat_mean = np.mean(all_b_hat, axis=0)
    a_hat_p05 = np.percentile(all_a_hat, 5, axis=0)
    a_hat_p95 = np.percentile(all_a_hat, 95, axis=0)
    b_hat_p05 = np.percentile(all_b_hat, 5, axis=0)
    b_hat_p95 = np.percentile(all_b_hat, 95, axis=0)    

    fig, ax = plt.subplots(2, 4, figsize=(14, 7), sharex=True) # Array Plot

    ax[0, 0].plot(x_true_base, 'k', lw=2, label='True Henon Map for x') # For x
    ax[0, 0].plot(x_hat_mean, 'b--', label='Mean EKF Estimate')
    ax[0, 0].plot(x_meas_base, 'r.', alpha=0.3, label='Measurements')
    ax[0, 0].set_title('x_k evolution')
    ax[0, 0].legend()

    ax[0, 1].plot(y_true_base, 'k', lw=2, label='True Henon Map for y') # For y
    ax[0, 1].plot(y_hat_mean, 'b--', label='Mean EKF Estimate')
    ax[0, 1].plot(y_meas_base, 'r.', alpha=0.3, label='Measurements')
    ax[0, 1].set_title('y_k evolution')
    ax[0, 1].legend()

    ax[0, 2].axhline(a_true, color='k', lw=2, label='True a') # For a
    ax[0, 2].plot(a_hat_mean, 'b--', label='Mean EKF Estimate')
    ax[0, 2].set_title('a Evolution Across MC Runs')
    ax[0, 2].set_ylabel('a')
    ax[0, 2].legend()

    ax[0, 3].axhline(b_true, color='k', lw=2, label='True b') # For b
    ax[0, 3].plot(b_hat_mean, 'b--', label='Mean EKF Estimate')
    ax[0, 3].set_title('b Evolution Across MC Runs')
    ax[0, 3].set_ylabel('b')
    ax[0, 3].legend()

    ax[1, 0].plot(mse_x, 'b-', label='MSE') # MC Evolution x
    ax[1, 0].axhline(0, color='k', linestyle='--', linewidth=1)
    ax[1, 0].set_xlabel('Iteration')
    ax[1, 0].set_ylabel('MSE')
    ax[1, 0].set_title('Evolution of x Over Time')
    ax[1, 0].legend()

    ax[1, 1].plot(mse_y, 'b-', label='MSE') # MC Evolution y
    ax[1, 1].axhline(0, color='k', linestyle='--', linewidth=1)
    ax[1, 1].set_xlabel('Iteration')
    ax[1, 1].set_ylabel('MSE')
    ax[1, 1].set_title('Evolution of y Over Time')
    ax[1, 1].legend()

    ax[1, 2].plot(variance_x, 'r-', label='MC Variance') # Filter Spread of x
    ax[1, 2].axhline(0, color='k', linestyle='--', linewidth=1)
    ax[1, 2].set_xlabel('Iteration')
    ax[1, 2].set_ylabel('Variance')
    ax[1, 2].set_title('Filter Spread of MC in x')
    ax[1, 2].legend()

    ax[1, 3].plot(variance_y, 'r-', label='MC Variance') # Filter Spread of y
    ax[1, 3].axhline(0, color='k', linestyle='--', linewidth=1)
    ax[1, 3].set_xlabel('Iteration')
    ax[1, 3].set_ylabel('Variance')
    ax[1, 3].set_title('Filter Spread of MC in y')
    ax[1, 3].legend()

    fig.suptitle(
        f"UKF for Henon Map, with a_true = {a_true:.2f}, b_true = {b_true:.2f}, Q = {Q_error:.1e}",
        fontsize=16)    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

interact( # Interactive Sliders
    interactive_ukf_henon,
    a_true=FloatSlider(value=1.4, min=0, max=1.42, step=0.01), # Max before Infinite Divergence
    b_true=FloatSlider(value=0.3, min=-0.3, max=0.31, step=0.01),
    Q_error=FloatSlider(value=1e-5, min=1e-12, max=1e-2,
                  step=1e-6, readout_format='.1e'),
    n_steps=IntSlider(value=200, min=100, max=500, step=50),
    alpha=FloatSlider(value=1e-3, min=1e-4, max=2e-1, step=1e-4, readout_format='.1e', description='Alpha'),
    beta=FloatSlider(value=2, min=0, max=4, step=0.1, description='Beta'),
    kappa=FloatSlider(value=1, min=-1, max=3, step=0.1, description='Kappa'))

interactive(children=(FloatSlider(value=1.4, description='a_true', max=1.42, step=0.01), FloatSlider(value=0.3…

<function __main__.interactive_ukf_henon(a_true=1.4, b_true=0.3, Q_error=1e-05, n_steps=100, alpha=0.001, beta=2, kappa=0)>